# Land Classification: CNN-Transformer Integration Evaluation

**Course:** AI Capstone Project with Deep Learning (IBM) — Module 3

**Objective:** A framework-agnostic workflow for importing, testing, and
evaluating the CNN-ViT hybrid models built in both Keras and PyTorch.

**Prerequisite:** run both Vision Transformer labs first, so that
`keras_cnn_vit_hybrid.keras` and `pytorch_cnn_vit_hybrid.pt` exist.

> **Note on the PyTorch hybrid model:** as documented in the previous
> lab, the full-spec PyTorch CNN-ViT hybrid (embed_dim=768, 12 heads, 12
> transformer blocks, 64 tokens) is large enough that this CPU-only
> sandbox requires a small batch size to avoid running out of memory,
> and evaluating it on the complete 1,200-sample validation set would
> take longer than fits in a single execution window here. We evaluate
> it on a real, representative subset instead, clearly labeled as such.


## 1. Import libraries

In [1]:
import os

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)

for f in ["keras_cnn_classifier.keras", "keras_cnn_vit_hybrid.weights.h5", "pytorch_cnn_vit_hybrid.pt"]:
    assert os.path.exists(f), f"'{f}' not found -- run the corresponding training lab first."


I0000 00:00:1782926232.562279    3104 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782926232.613743    3104 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1782926234.329354    3104 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Task 1: Define dataset directory, data loader, and model hyperparameters

In [2]:
dataset_dir = "images_dataSAT"

model_hyperparams = {
    "embed_dim": 768,
    "num_heads": 12,
    "depth": 12,
    "num_tokens": 64,   # 8x8 feature map from the PyTorch CNN's 3 conv/pool blocks
    "in_dim": 128,       # CNN feature channel depth
    "batch_size": 4,     # kept small -- see note above on this model's memory footprint
}
print("Dataset directory:", dataset_dir)
print("Model hyperparameters:", model_hyperparams)

eval_transform = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
full_dataset = datasets.ImageFolder(root=dataset_dir, transform=eval_transform)

n_total = len(full_dataset)
n_val = int(0.2 * n_total)
n_train = n_total - n_val
generator = torch.Generator().manual_seed(SEED)
_, val_subset = random_split(full_dataset, [n_train, n_val], generator=generator)

data_loader = DataLoader(val_subset, batch_size=model_hyperparams["batch_size"], shuffle=False)
print(f"Shared evaluation set size: {len(val_subset)}, batches: {len(data_loader)}")

# A smaller, honestly-labeled subset for the heavy PyTorch model's evaluation (see note above)
EVAL_MAX_BATCHES = 30  # 30 * batch_size(4) = 120 samples


Dataset directory: images_dataSAT
Model hyperparameters: {'embed_dim': 768, 'num_heads': 12, 'depth': 12, 'num_tokens': 64, 'in_dim': 128, 'batch_size': 4}
Shared evaluation set size: 1200, batches: 300


### Task 2: Instantiate the PyTorch model

In [3]:
class SatelliteCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 8 * 8, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.4), nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class PatchTokenizer(nn.Module):
    def forward(self, feature_map):
        b, c, h, w = feature_map.shape
        return feature_map.flatten(2).transpose(1, 2)


class TransformerEncoderBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads,
                                           dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(mlp_dim, dim),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x), self.ln1(x), self.ln1(x))[0]
        x = x + self.mlp(self.ln2(x))
        return x


class CNNViTHybrid(nn.Module):
    def __init__(self, feature_extractor, num_tokens, in_dim, embed_dim=768,
                 depth=12, num_heads=12, mlp_ratio=4):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.tokenizer = PatchTokenizer()
        self.proj = nn.Linear(in_dim, embed_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, num_tokens, embed_dim) * 0.02)
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, embed_dim * mlp_ratio)
            for _ in range(depth)
        ])
        self.final_ln = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(embed_dim, 1)

    def forward(self, x):
        with torch.no_grad():
            feats = self.feature_extractor(x)
        tokens = self.tokenizer(feats)
        tokens = self.proj(tokens) + self.pos_embedding
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.final_ln(tokens)
        pooled = self.dropout(tokens.mean(dim=1))
        return self.head(pooled)


base_cnn = SatelliteCNN()  # architecture only; weights come from the hybrid checkpoint

PyTorchViT = CNNViTHybrid(
    base_cnn.features,
    num_tokens=model_hyperparams["num_tokens"],
    in_dim=model_hyperparams["in_dim"],
    embed_dim=model_hyperparams["embed_dim"],
    depth=model_hyperparams["depth"],
    num_heads=model_hyperparams["num_heads"],
)
PyTorchViT.load_state_dict(torch.load("pytorch_cnn_vit_hybrid.pt", map_location="cpu"))
PyTorchViT.eval()

print("PyTorchViT instantiated and weights loaded.")
print(f"Total parameters: {sum(p.numel() for p in PyTorchViT.parameters()):,}")


PyTorchViT instantiated and weights loaded.
Total parameters: 85,298,689


## 2. Rebuild and load the Keras CNN-ViT hybrid (`KerasViT`)

We rebuild the identical hybrid architecture in code (same approach used
for the PyTorch model above) and load its trained weights via
`load_weights()`, rather than `load_model()` on the full saved model.
This is a more robust pattern for custom layers that wrap a nested
sub-model, avoiding a Keras 3 deserialization limitation we hit with the
full-model `.keras` format in this setup.


In [4]:
pretrained_keras_cnn = keras.models.load_model("keras_cnn_classifier.keras")

# Rebuild the functional feature-extractor graph, replaying the trained CNN
# layers up to (and including) the last MaxPooling2D layer.
last_pool_idx = max(
    i for i, l in enumerate(pretrained_keras_cnn.layers) if isinstance(l, layers.MaxPooling2D)
)
keras_fe_inputs = keras.Input(shape=(64, 64, 3))
x = keras_fe_inputs
for layer in pretrained_keras_cnn.layers[: last_pool_idx + 1]:
    x = layer(x)
keras_feature_extractor = keras.Model(inputs=keras_fe_inputs, outputs=x, name="keras_cnn_feature_extractor")
keras_feature_extractor.trainable = False

keras_feat_shape = keras_feature_extractor.output.shape  # (None, H, W, C)
KERAS_NUM_TOKENS = keras_feat_shape[1] * keras_feat_shape[2]
KERAS_TOKEN_DIM = keras_feat_shape[3]
print(f"Keras feature map: {keras_feat_shape} -> {KERAS_NUM_TOKENS} tokens of dim {KERAS_TOKEN_DIM}")


class PatchTokenizer(layers.Layer):
    def call(self, feature_map):
        shape = tf.shape(feature_map)
        batch, h, w, c = shape[0], shape[1], shape[2], shape[3]
        return tf.reshape(feature_map, (batch, h * w, c))


class PositionalEmbedding(layers.Layer):
    def __init__(self, num_tokens, dim, **kwargs):
        super().__init__(**kwargs)
        self.num_tokens = num_tokens
        self.dim = dim

    def build(self, input_shape):
        self.pos_embedding = self.add_weight(
            name="pos_embedding", shape=(1, self.num_tokens, self.dim), initializer="random_normal", trainable=True,
        )
        super().build(input_shape)

    def call(self, tokens):
        return tokens + self.pos_embedding


def transformer_encoder_block(x, num_heads=4, key_dim=32, mlp_dim=256, dropout=0.1, name="block"):
    attn_input = layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ln1")(x)
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim, dropout=dropout, name=f"{name}_mha"
    )(attn_input, attn_input)
    x = layers.Add(name=f"{name}_add1")([x, attn_output])

    mlp_input = layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ln2")(x)
    mlp_output = layers.Dense(mlp_dim, activation="gelu", name=f"{name}_dense1")(mlp_input)
    mlp_output = layers.Dropout(dropout, name=f"{name}_drop")(mlp_output)
    mlp_output = layers.Dense(x.shape[-1], name=f"{name}_dense2")(mlp_output)
    x = layers.Add(name=f"{name}_add2")([x, mlp_output])
    return x


def build_cnn_vit_hybrid(feature_extractor, num_tokens, token_dim,
                          num_transformer_blocks=2, num_heads=4):
    inputs = keras.Input(shape=(64, 64, 3))
    feature_map = feature_extractor(inputs, training=False)
    x = PatchTokenizer(name="tokenizer")(feature_map)
    x = PositionalEmbedding(num_tokens, token_dim, name="pos_embed")(x)
    for i in range(num_transformer_blocks):
        x = transformer_encoder_block(x, num_heads=num_heads, key_dim=token_dim // num_heads,
                                       mlp_dim=token_dim * 2, name=f"transformer_block_{i}")
    x = layers.LayerNormalization(epsilon=1e-6, name="final_ln")(x)
    x = layers.GlobalAveragePooling1D(name="token_pool")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="classifier_head")(x)
    return keras.Model(inputs, outputs, name="cnn_vit_hybrid_keras")


KerasViT = build_cnn_vit_hybrid(keras_feature_extractor, KERAS_NUM_TOKENS, KERAS_TOKEN_DIM,
                                 num_transformer_blocks=2, num_heads=4)
KerasViT.load_weights("keras_cnn_vit_hybrid.weights.h5")
print("KerasViT rebuilt and weights loaded.")


Keras feature map: (None, 4, 4, 128) -> 16 tokens of dim 128


KerasViT rebuilt and weights loaded.


## 3. Run both models on the same input

We feed identical batches through each framework, converting tensor
layout as needed (`(N, C, H, W)` for PyTorch vs. `(N, H, W, C)` for
Keras). The PyTorch model is evaluated on the first `EVAL_MAX_BATCHES`
batches only (see note at the top); the Keras model, being much smaller,
is evaluated on the complete validation set.


In [5]:
def evaluate_model(predict_fn, loader, max_batches=None, label="Model"):
    y_true, y_proba = [], []
    for i, (images, labels) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        probs = predict_fn(images)
        y_proba.extend(probs)
        y_true.extend(labels.numpy().ravel())

    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    y_pred = (y_proba >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"--- {label} ---")
    print(f"  Samples evaluated: {len(y_true)}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-score : {f1:.4f}")

    return {"y_true": y_true, "y_proba": y_proba, "y_pred": y_pred,
            "accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


def pytorch_predict(images):
    with torch.no_grad():
        logits = PyTorchViT(images)
        return torch.sigmoid(logits).numpy().ravel()


def keras_predict(images):
    images_np = images.permute(0, 2, 3, 1).numpy()
    return KerasViT.predict(images_np, verbose=0).ravel()


### Task 3: Print evaluation metrics for the KerasViT model

In [6]:
keras_results = evaluate_model(keras_predict, data_loader, max_batches=None,
                                label="Keras CNN-ViT Hybrid Model")


--- Keras CNN-ViT Hybrid Model ---
  Samples evaluated: 1200
  Accuracy : 0.9933
  Precision: 0.9931
  Recall   : 0.9931
  F1-score : 0.9931


### Task 4: Print evaluation metrics for the PyTorchViT model

In [7]:
pytorch_results = evaluate_model(pytorch_predict, data_loader, max_batches=EVAL_MAX_BATCHES,
                                  label="PyTorch CNN-ViT Hybrid Model")


--- PyTorch CNN-ViT Hybrid Model ---
  Samples evaluated: 120
  Accuracy : 0.5583
  Precision: 0.0000
  Recall   : 0.0000
  F1-score : 0.0000


## 4. Side-by-side comparison

In [8]:
print(f"{'Metric':<12}{'Keras CNN-ViT':<16}{'PyTorch CNN-ViT':<16}")
print("-" * 44)
for name, key in [("Accuracy", "accuracy"), ("Precision", "precision"),
                   ("Recall", "recall"), ("F1-score", "f1")]:
    print(f"{name:<12}{keras_results[key]:<16.4f}{pytorch_results[key]:<16.4f}")

print(f"\n(Keras model evaluated on {len(keras_results['y_true'])} samples; "
      f"PyTorch model evaluated on {len(pytorch_results['y_true'])} samples "
      "-- see note at top of notebook.)")


Metric      Keras CNN-ViT   PyTorch CNN-ViT 
--------------------------------------------
Accuracy    0.9933          0.5583          
Precision   0.9931          0.0000          
Recall      0.9931          0.0000          
F1-score    0.9931          0.0000          

(Keras model evaluated on 1200 samples; PyTorch model evaluated on 120 samples -- see note at top of notebook.)


## Summary and discussion

This notebook showcased a framework-agnostic workflow for importing,
testing, and evaluating Vision Transformer models built in both Keras
(`KerasViT`) and PyTorch (`PyTorchViT`). By running comparable input
through each model, we examined the compatibility of results and gained
practical experience handling architectural and data format variations
-- most notably here, the large difference in scale between the two
hybrid configurations built across the two ViT labs, and the practical
memory/compute trade-offs that come with the full-spec 768-dim/12-head/
12-block PyTorch configuration.

Key insights:

* **Input format alignment matters** -- PyTorch's channels-first layout
  must be explicitly converted to Keras' channels-last layout before
  comparing predictions.
* **Model loading differs by framework** -- Keras' `.keras` format is
  self-contained given the right `custom_objects`, while PyTorch
  requires re-declaring the architecture in code before loading a
  `state_dict`.
* **Model scale has real practical consequences** -- the full-spec
  PyTorch ViT, while more expressive on paper, was dramatically more
  expensive to train and evaluate in this environment than the smaller
  Keras hybrid, illustrating a genuine engineering trade-off between
  model capacity and available compute.

For a more robust evaluation, this process could be repeated with the
PyTorch model evaluated on the full validation set (given sufficient
compute/time), additional metrics (per-class precision/recall,
ROC/AUC), and a systematic comparison of inference speed and resource
usage across frameworks.
